# Instagram



In [ ]:
pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29

In [ ]:
pip install numpy==1.26.4

In [ ]:
from crewai import Agent, Task, Crew
from crewai_tools import tool

In [ ]:
import openai

openai.api_key = " "

In [ ]:
import os
#from utils import get_openai_api_key

#openai_api_key = get_openai_api_key()
import os
os.environ["OPENAI_API_KEY"] = " "
os.environ["OPENAI_MODEL_NAME"] = 'gpt-4o-mini'

In [ ]:
#Instagram Content

In [ ]:
import os
import requests
from crewai import Agent, Task, Crew
from crewai_tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
import re
from IPython.display import Image, display

openai_api_key = " "

llm = ChatOpenAI(
    openai_api_base="https://api.openai.com/v1",
    openai_api_key=openai_api_key,
    model_name="gpt-4o-mini"
)

@tool
def generateimage(query: str) -> str:
    """
    Generates an image using DALL-E based on chapter content and character details.
    Displays the image instead of downloading it.
    """
    client = OpenAI(api_key=openai_api_key)  # Use OpenAI API key directly

    response = client.images.generate(
        model="dall-e-3",
        prompt=f"Create a realistic image of: {query}. Style: Focus on lifelike details, accurate lighting, and natural textures, with a realistic color palette. The scene should capture the true essence of the description, ensuring it looks as if it could exist in the real world.",
        size="1024x1024",
        quality="standard",
        n=1,
    )
    image_url = response.data[0].url


    display(Image(url=image_url))

    return image_url

#Agent 1: Caption & Hashtag Strategist
caption_agent = Agent(
    name='Caption & Hashtag Strategist',
    role='Social Media Expert',
    goal='Create engaging captions and effective hashtag strategies',
    backstory="You're working as a social media copywriter who specializes in Instagram content. "
              "Your goal is to craft engaging captions that resonate with audiences emotionally or humorously on {theme}, "
              "while also ensuring discoverability through relevant hashtags. "
              "You analyze trends and write content that aligns with the visual's intent and the creator's brand voice. "
              "Your work is basis the image generated by the Visual Content Creator.",
    allow_delegation=False,
    verbose=True
)

# Agent to generate images
visual_agent = Agent(
   name='Visual Content Creator',
   role='AI Image Generator',
   goal="Generate visually compelling and on-brand images for Instagram content based on the theme: {theme}",
  backstory="You're an AI-based visual artist creating Instagram-ready images based on the theme: {theme}. "
              "You work in collaboration with the Caption & Hashtag Strategist, who builds engagement-driven captions around your visuals. "
              "Your images are used to attract attention, convey mood, and support the messaging strategy. "
              "You use cutting-edge generative AI (like DALL·E) to create realistic, vibrant, and aesthetically pleasing images whoch look real and not AI generated. "
              "You ensure that your visuals are aligned with Instagram design sensibilities, have high visual impact, and inspire interaction. "
              "Your creations are original, style-consistent, and relevant to the target theme.",
   verbose=True,
   llm=llm,
   tools=[generateimage],  # Tool to generate images using DALL-E API
  #allow_delegation=False
)

# Task 1: Generate Captions & Hashtags
task_captions = Task(
      description="Generate a catchy Instagram caption and a set of trending hashtags for a given theme.",
      agent=caption_agent,
      expected_output="A catchy caption and relevant hashtags."  # Added expected_output
)
task_captions_result = task_captions.execute()

# Task to generate images
task_image = Task(
     description="Generate a beautiful Instagram-style visual based on the theme and look natural and not AI generated.",
     agent=visual_agent,
     expected_output='A high-quality image matching the chapter description and tone.',
)

# Create the Crew
crew = Crew(
  agents=[caption_agent, visual_agent],
  tasks=[task_captions, task_image],
  verbose=True,
)

# Start the task
result = crew.kickoff(inputs={"theme": "cafe"})
print(result)


In [ ]:
from IPython.display import Markdown
Markdown(result)